# Nsquare | YOLOv1 객체 탐지 실습

**목표:** PASCAL VOC 2012 이미지로 객체 탐지 모델을 직접 학습하고, **정량 평가(mAP@0.5)**와 **정성 평가(예측 Bounding Box 시각화)**를 수행한 뒤, 여러 학습 설정을 비교해 성능을 높여 봅니다.

**실행 순서:** `런타임 → 모두 실행`을 누르면 **기본 학습 → Validation 정량·정성 평가**가 실행됩니다. 추가 실험은 아래 `RUN_EXTRA_EXPERIMENTS`를 켜서 진행할 수 있습니다. **최종 Test**는 설정을 확정한 후 마지막 셀의 `RUN_FINAL_TEST=True`로 **한 번만** 수행합니다.

> Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU(T4 이상 권장)**을 선택하세요. 전체 VOC 데이터를 448×448로 학습하므로 GPU가 필요합니다. 학습 설정 및 실험 후보를 바꾸는 부분만 편집하면 됩니다.

## 1. 환경 및 데이터 준비

- 데이터: 제공된 **PASCAL VOC 2012** 압축 파일 (원본 링크 유지)
- 대상 클래스: `bicycle`, `bus`, `car`, `cat`, `dog`, `person`
- 입력 크기: `448×448`, ImageNet 정규화
- 분할: 공식 `train.txt`의 6개 클래스 포함 이미지를 **80% Train / 20% Validation** (seed=42)
- **최종 Test:** 공식 `val.txt` 전부. 실험 중에는 사용하지 않습니다.

In [ ]:
# Colab에서 기본 제공되지 않은 패키지만 설치합니다.
import importlib.util
import subprocess
import sys

need = {'cv2': 'opencv-python-headless', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm', 'pandas': 'pandas'}
missing = [package for module, package in need.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import os, random, shutil, zipfile, copy, gc
from pathlib import Path
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.datasets import VOCDetection

SEED = 42

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('Device :', device, '| CUDA GPU:', torch.cuda.get_device_name(0) if device.type == 'cuda' else '없음 (GPU 사용 권장)')

In [ ]:
# 제공된 원본 데이터 URL 및 주석 파일을 그대로 사용합니다.
DATA_URL = 'https://o365uos-my.sharepoint.com/:u:/g/personal/hjseong_office_uos_ac_kr/EQPy0vqC_-5Lj7G6Z_tslc0BsjvqgZ0LGymvuix3T4WGeQ?e=hqy88B&download=1'
DATA_DIR = Path('./dataset')
ARCHIVE = DATA_DIR / 'VOC2012_train_val.zip'
VOC_DIR = DATA_DIR / 'VOCdevkit' / 'VOC2012'
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (VOC_DIR / 'ImageSets' / 'Main' / 'train.txt').is_file():
    if not ARCHIVE.is_file() or not zipfile.is_zipfile(ARCHIVE):
        print('원본 VOC2012 데이터 다운로드 시도...')
        try:
            subprocess.run(['wget', '-q', '--show-progress', '--tries=3', '--timeout=30',
                            DATA_URL, '-O', str(ARCHIVE)], check=True)
            if not zipfile.is_zipfile(ARCHIVE):
                raise RuntimeError('다운로드된 파일이 유효한 ZIP이 아닙니다.')
        except Exception as e:
            if ARCHIVE.exists():
                ARCHIVE.unlink()
            print('자동 다운로드 실패:', e)
            print('원본 VOC2012_train_val.zip 파일을 직접 업로드해 주세요.')
            try:
                from google.colab import files
                uploaded = files.upload()
                if 'VOC2012_train_val.zip' not in uploaded:
                    raise FileNotFoundError('원본 VOC2012_train_val.zip 파일이 필요합니다.')
                ARCHIVE.write_bytes(uploaded['VOC2012_train_val.zip'])
            except ImportError as exc:
                raise RuntimeError(f'제공된 원본 ZIP을 {ARCHIVE} 위치에 넣은 뒤 재실행하세요.') from exc
    assert zipfile.is_zipfile(ARCHIVE), '유효한 원본 VOC ZIP 파일이 필요합니다.'
    print('압축 해제 중...')
    with zipfile.ZipFile(ARCHIVE, 'r') as zf:
        zf.extractall(DATA_DIR)

    # 제공된 압축파일의 폴더 구조가 다르더라도 동일 VOC2012 디렉터리로 배치합니다.
    if not (VOC_DIR / 'ImageSets' / 'Main' / 'train.txt').is_file():
        candidates = [p.parent for p in DATA_DIR.rglob('JPEGImages')
                      if p.is_dir() and (p.parent / 'Annotations').is_dir()
                      and (p.parent / 'ImageSets' / 'Main' / 'train.txt').is_file()]
        if not candidates:
            raise FileNotFoundError('압축파일에서 VOC2012/JPEGImages, Annotations, ImageSets/Main/train.txt를 찾을 수 없습니다.')
        src_dir = candidates[0]
        if src_dir.resolve() != VOC_DIR.resolve():
            VOC_DIR.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src_dir), str(VOC_DIR))

assert (VOC_DIR / 'ImageSets' / 'Main' / 'train.txt').is_file(), 'VOC2012 train.txt를 찾을 수 없습니다.'
assert (VOC_DIR / 'ImageSets' / 'Main' / 'val.txt').is_file(), 'VOC2012 val.txt를 찾을 수 없습니다.'
DATA_PATH = str(DATA_DIR)
print('데이터 준비 완료:', VOC_DIR)

### 데이터 전처리와 Target 변환

원본의 클래스 필터링, YOLO Target 인코딩 `(7, 7, 11)`, Validation/Test 전처리, 고정 분할은 그대로 사용합니다. **데이터 실험에서 변경할 수 있는 부분은 Train 이미지에 적용되는 증강 강도뿐**입니다.

In [ ]:
# 원본 데이터 클래스 및 인코딩 구현: target encoding / validation preprocess 변경 금지.

import xml.etree.ElementTree as ET

# We use 6 classes in PASCAL VOC dataset
VOC_CLASSES = (
    'bicycle', 'bus', 'car', 'cat', 'dog', 'person'
    # 'aeroplane', 'bird', 'boat', 'bottle',
    # 'chair', 'cow', 'diningtable', 'horse',
    # 'motorbike', 'pottedplant',
    # 'sheep', 'sofa', 'train', 'tvmonitor'
)

CLASS_TO_IDX = {name: idx for idx, name in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES) # Should be 6
TARGET_CLASSES_SET = set(VOC_CLASSES)

# This class wraps VOCDetection to filter out images (20 classes -> 6 classes)
# that do NOT contain any of the classes in `target_classes`
class FilteredVOCDetection(VOCDetection):
    def __init__(self, root, year, image_set, download, target_classes: set, transforms=None):

        # Initialize the parent class, but WITHOUT transforms
        # We will apply transforms manually in __getitem__
        super().__init__(root=root, year=year, image_set=image_set, download=download, transforms=None)

        self.target_classes = target_classes
        # Store the joint transform
        self.joint_transform = transforms

        print(f"Filtering dataset '{image_set}' for {len(self.target_classes)} target classes...")
        self._filter_dataset()
        # self.images is overwritten, so len(self.images) will be correct
        print(f"Filtering complete. Found {len(self.images)} valid images out of original count.")

    def _filter_dataset(self):
        # self.annotations is a property, self.images is a list
        valid_indices = []
        original_image_paths = self.images
        original_annot_paths = self.annotations # Read from the property

        for i in tqdm(range(len(original_annot_paths)), desc="Filtering Images"):
            annot_path = original_annot_paths[i]

            try:
                tree = ET.parse(annot_path)
                root = tree.getroot()
                found = False

                for obj in root.findall('object'):
                    class_name = obj.find('name').text
                    if class_name in self.target_classes:
                        found = True
                        break # Found a valid object, no need to check others

                if found:
                    valid_indices.append(i)
            except Exception as e:
                img_name = os.path.basename(original_image_paths[i])
                print(f"Warning: Could not parse {annot_path} (image: {img_name}). Error: {e}. Skipping image.")

        # Overwrite the parent class's image list
        # This works because self.images is a simple list attribute
        self.images = [original_image_paths[i] for i in valid_indices]

        # Store the filtered list in a new attribute
        self.filtered_annotations = [original_annot_paths[i] for i in valid_indices]

    def __getitem__(self, index):
        # This implementation is copied from the parent VOCDetection.__getitem__
        # to ensure we load the raw data

        # self.images was successfully overwritten, so this is correct
        img = Image.open(self.images[index]).convert('RGB')

        # Read from filtered_annotations list instead of the parent's property
        target = self.parse_voc_xml(ET.parse(self.filtered_annotations[index]).getroot())

        # Now, apply the joint transform that we stored
        if self.joint_transform:
            img, target = self.joint_transform(img, target)

        return img, target

# Converts VOCDetection's XML target to YOLO's (S, S, C+5) tensor.
class EncodeYoloTarget:
    def __init__(self, S=7, C=6):
        self.S = S
        self.C = C
        self.target_tensor_shape = (S, S, C + 5) # (x, y, w, h, conf) + C_classes

    def __call__(self, target):
        # target: annotation dictionary returned by VOCDetection
        target_tensor = torch.zeros(self.target_tensor_shape)

        try:
            img_w = int(target['annotation']['size']['width'])
            img_h = int(target['annotation']['size']['height'])

            # Objects are returned as a list if multiple, or a dict if single
            try:
                objs = target['annotation']['object']
                if not isinstance(objs, list):
                    objs = [objs]
            except KeyError:
                objs = [] # Handle images with no objects

            for obj in objs:
                class_name = obj['name']

                # This check uses the global CLASS_TO_IDX
                if class_name not in CLASS_TO_IDX:
                    continue # Skip objects that are not in our 6 target classes

                class_idx = CLASS_TO_IDX[class_name]

                bbox = obj['bndbox']
                # Use float for more precision before int casting
                xmin = float(bbox['xmin'])
                ymin = float(bbox['ymin'])
                xmax = float(bbox['xmax'])
                ymax = float(bbox['ymax'])

                # Skip invalid boxes (e.g., after augmentation)
                if xmax <= xmin or ymax <= ymin:
                    continue

                # Convert to YOLO format (x_center, y_center, w, h) - relative to image (0~1)
                x_center = (xmin + xmax) / 2 / img_w
                y_center = (ymin + ymax) / 2 / img_h
                w = (xmax - xmin) / img_w
                h = (ymax - ymin) / img_h

                # Calculate cell index
                grid_x_idx = int(x_center * self.S)
                grid_y_idx = int(y_center * self.S)

                # Clamp values to be within grid [0, S-1]
                # This handles edge cases where x_center or y_center is exactly 1.0
                grid_x_idx = min(grid_x_idx, self.S - 1)
                grid_y_idx = min(grid_y_idx, self.S - 1)

                # Coordinates within the cell (0~1)
                x_cell = (x_center * self.S) - grid_x_idx
                y_cell = (y_center * self.S) - grid_y_idx

                # Check if this cell is already responsible for an object (YOLOv1: one object per cell)
                if target_tensor[grid_y_idx, grid_x_idx, self.C] == 0:
                    # Confidence
                    target_tensor[grid_y_idx, grid_x_idx, self.C] = 1
                    # Bbox (x_cell, y_cell, w_img, h_img)
                    target_tensor[grid_y_idx, grid_x_idx, (self.C + 1):(self.C + 5)] = torch.tensor([x_cell, y_cell, w, h])
                    # Class
                    target_tensor[grid_y_idx, grid_x_idx, class_idx] = 1

        except Exception as e:
            # Print the specific target that failed, which is more useful
            img_filename = target.get('annotation', {}).get('filename', 'Unknown')
            print(f"Warning: Error parsing target for {img_filename}. Error: {e}")
            # Return an empty tensor
            return torch.zeros(self.target_tensor_shape)

        return target_tensor

# Handles augmentations that must be applied to both image and target (bboxes)
class JointTransform:
    def __init__(self, S=7, C=6, input_size=(448, 448), augment=True,
                 scale_range_aug: tuple = (0.7, 1.3)):
        self.S = S
        self.C = C
        self.input_size = input_size
        self.augment = augment

        # The range for random scaling (e.g., 0.7x to 1.3x)
        self.scale_range_aug = scale_range_aug
        # The fill color for empty areas after translation (128, 128, 128) is gray
        self.padding_fill = (128, 128, 128)

        # This is the target encoder
        self.target_encoder = EncodeYoloTarget(S=S, C=C)

        # Augmentation transforms (image only)
        self.color_jitter = transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1)

        # Final image transforms (applied after augmentations)
        self.final_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __call__(self, img_pil: Image.Image, target_dict: dict):
        # img_pil: PIL.Image
        # target_dict: annotation dictionary

        # Get original image size
        original_w = int(target_dict['annotation']['size']['width'])
        original_h = int(target_dict['annotation']['size']['height'])

        # Get original boxes
        # It's easier to work with a simple list of [xmin, ymin, xmax, ymax, name]
        original_boxes = []
        try:
            objs = target_dict['annotation']['object']
            if not isinstance(objs, list):
                objs = [objs] # Handle single object case
            for obj in objs:
                bbox = obj['bndbox']
                original_boxes.append([
                    int(float(bbox['xmin'])), int(float(bbox['ymin'])),
                    int(float(bbox['xmax'])), int(float(bbox['ymax'])),
                    obj['name']
                ])
        except KeyError:
            pass # No objects in this image

        # Apply Augmentations (if enabled)
        if self.augment:
            # Color Jitter (Image only)
            img_pil = self.color_jitter(img_pil)

            # Random Scaling and Translation (Jitter)
            # This logic replaces the simple deterministic resize

            # Get a random scale factor
            scale = random.uniform(self.scale_range_aug[0], self.scale_range_aug[1])
            scaled_w = int(original_w * scale)
            scaled_h = int(original_h * scale)

            # Resize the image
            img_pil = img_pil.resize((scaled_w, scaled_h), Image.BILINEAR)

            # Create a new canvas (e.g., 448x448) filled with gray
            canvas_w, canvas_h = self.input_size
            canvas = Image.new('RGB', self.input_size, self.padding_fill)

            # Calculate random translation (dx, dy)
            # This determines where the top-left corner of the scaled image is placed
            dx = int(random.uniform(min(0, canvas_w - scaled_w), max(0, canvas_w - scaled_w)))
            dy = int(random.uniform(min(0, canvas_h - scaled_h), max(0, canvas_h - scaled_h)))

            # Paste the scaled image onto the canvas
            canvas.paste(img_pil, (dx, dy))
            img_pil = canvas # This canvas is now our augmented image

            # Update boxes for BOTH scaling and translation
            scaled_translated_boxes = []
            for (xmin, ymin, xmax, ymax, class_name) in original_boxes:
                # Apply scaling
                xmin_s, ymin_s = xmin * scale, ymin * scale
                xmax_s, ymax_s = xmax * scale, ymax * scale

                # Apply translation
                xmin_t, ymin_t = xmin_s + dx, ymin_s + dy
                xmax_t, ymax_t = xmax_s + dx, ymax_s + dy

                # Clip boxes to the canvas boundaries [0, 0, canvas_w, canvas_h]
                xmin_c = max(0, xmin_t)
                ymin_c = max(0, ymin_t)
                xmax_c = min(canvas_w, xmax_t)
                ymax_c = min(canvas_h, ymax_t)

                # Filter boxes that have zero area after clipping
                if xmax_c > xmin_c and ymax_c > ymin_c:
                    scaled_translated_boxes.append([xmin_c, ymin_c, xmax_c, ymax_c, class_name])

            # Random Horizontal Flip
            final_boxes = []
            if random.random() < 0.5:
                img_pil = TF.hflip(img_pil)

                for (xmin, ymin, xmax, ymax, class_name) in scaled_translated_boxes:
                    # Flipped xmin is (canvas_width - original xmax)
                    # Flipped xmax is (canvas_width - original xmin)
                    flipped_xmin = canvas_w - xmax
                    flipped_xmax = canvas_w - xmin
                    final_boxes.append([flipped_xmin, ymin, flipped_xmax, ymax, class_name])
            else:
                final_boxes = scaled_translated_boxes

            # Rebuild target_dict
            # The target_dict must now reflect the new 448x448 image
            target_dict['annotation']['size']['width'] = str(canvas_w)
            target_dict['annotation']['size']['height'] = str(canvas_h)

            new_obj_list = []
            for (xmin, ymin, xmax, ymax, class_name) in final_boxes:
                new_obj_list.append({
                    'name': class_name,
                    'bndbox': {
                        'xmin': str(int(xmin)),
                        'ymin': str(int(ymin)),
                        'xmax': str(int(xmax)),
                        'ymax': str(int(ymax))
                    }
                })
            target_dict['annotation']['object'] = new_obj_list

        # Deterministic Resize (for Validation/Test)
        else:
            # If not augmenting, just do the original deterministic resize
            img_pil = TF.resize(img_pil, self.input_size)
            new_w, new_h = self.input_size

            # Calculate scale factors
            w_scale = new_w / original_w
            h_scale = new_h / original_h

            # Update target size AND scale bounding boxes
            target_dict['annotation']['size']['width'] = str(new_w)
            target_dict['annotation']['size']['height'] = str(new_h)

            # We must rebuild the dictionary here too, to use the original_boxes
            new_obj_list = []
            for (xmin, ymin, xmax, ymax, class_name) in original_boxes:
                # Scale coordinates and clamp to new boundaries
                xmin_s = max(0, min(new_w, int(xmin * w_scale)))
                ymin_s = max(0, min(new_h, int(ymin * h_scale)))
                xmax_s = max(0, min(new_w, int(xmax * w_scale)))
                ymax_s = max(0, min(new_h, int(ymax * h_scale)))

                if xmax_s > xmin_s and ymax_s > ymin_s:
                    new_obj_list.append({
                        'name': class_name,
                        'bndbox': {
                            'xmin': str(xmin_s),
                            'ymin': str(ymin_s),
                            'xmax': str(xmax_s),
                            'ymax': str(ymax_s)
                        }
                    })
            target_dict['annotation']['object'] = new_obj_list


        # Final Image Transform (to Tensor)
        img_tensor = self.final_transform(img_pil)

        # Target Transform (to Tensor)
        # Now, target_encoder receives coordinates relative to the 448x448 image
        # It will also filter out non-target classes
        target_tensor = self.target_encoder(target_dict)

        return img_tensor, target_tensor

In [ ]:
# 원본 학습 데이터에서 Seed 42로 고정 Train/Validation 분할

DATA_PATH = "./dataset"
INPUT_SIZE = (448, 448)
S, B, C = 7, 2, len(VOC_CLASSES)

train_transform = JointTransform(S=S, C=C, input_size=INPUT_SIZE, augment=True)
validation_transform = JointTransform(S=S, C=C, input_size=INPUT_SIZE, augment=False)

train_full_aug = FilteredVOCDetection(
    root=DATA_PATH, year="2012", image_set="train", download=False,
    target_classes=TARGET_CLASSES_SET, transforms=train_transform,
)
train_full_eval = FilteredVOCDetection(
    root=DATA_PATH, year="2012", image_set="train", download=False,
    target_classes=TARGET_CLASSES_SET, transforms=validation_transform,
)
assert train_full_aug.images == train_full_eval.images, "Training and validation image order must match."

split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_full_aug), generator=split_generator).tolist()
num_train = int(0.80 * len(permutation))
train_indices = permutation[:num_train]
val_indices = permutation[num_train:]

train_dataset = Subset(train_full_aug, train_indices)
validation_dataset = Subset(train_full_eval, val_indices)
val_image_ids = [Path(train_full_eval.images[i]).stem for i in val_indices]

assert len(set(train_indices) & set(val_indices)) == 0
assert len(set(val_image_ids)) == len(val_image_ids)

print("Training images  :", len(train_dataset))
print("Validation images:", len(validation_dataset))
print("Test images      : reserved for the FINAL TEST EVALUATION cell")
print("Random seed      :", SEED)

In [ ]:
# 데이터가 YOLO 입력 및 정답 형식에 맞게 구성됐는지 확인합니다.
example_image, example_target = train_dataset[0]
assert example_image.shape == (3, 448, 448)
assert example_target.shape == (7, 7, 11)
print('Train / Validation:', len(train_dataset), '/', len(validation_dataset))
print('이미지 텐서:', tuple(example_image.shape))
print('정답 텐서 :', tuple(example_target.shape), '| 객체가 들어있는 셀 수:', int(example_target[..., C].sum()))

## 2. YOLOv1 모델과 Loss

사용하는 모델은 **ImageNet-pretrained VGG16 → Conv 4개 → FC(4096) → FC(784)**이며, 출력은 `(batch, 7, 7, 16)`입니다. 한 Grid Cell마다 **6개 클래스 점수 + 2개 Bounding Box × (x, y, w, h, confidence)**를 예측합니다.

YOLOv1 Loss는 **분류 / 좌표 / 객체 confidence / 배경 confidence** 네 항으로 구성합니다. 객체가 포함된 Cell에서는 정답 Box와 IoU가 가장 높은 예측 Box가 책임을 맡습니다. `λcoord=5`, `λnoobj=0.5`는 고정합니다.

In [ ]:
class YOLOv1(nn.Module):
    def __init__(self, S=7, B=2, C=6, pretrained_backbone=True):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        weights = models.VGG16_Weights.IMAGENET1K_V1 if pretrained_backbone else None
        self.features = models.vgg16(weights=weights).features

        # 원본 명세: 4개의 3x3 Conv, 두 번째 Conv에서만 stride=2
        self.conv_extension = nn.Sequential(
            nn.Conv2d(512, 1024, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.fc = nn.Sequential(
            nn.Linear(1024 * 7 * 7, 4096),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, S * S * (C + B * 5)),
        )
        self._initialize_weights()

    def _initialize_weights(self):
        # 원본 초기화: 추가 Conv=Kaiming, FC=Xavier (VGG 사전학습 가중치는 유지)
        for m in self.conv_extension.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        for m in self.fc.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)                 # (N, 512, 14, 14)
        x = self.conv_extension(x)          # (N, 1024, 7, 7)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x.reshape(-1, self.S, self.S, self.C + self.B * 5)


class YoloLoss(nn.Module):
    def __init__(self, S=7, B=2, C=6, lambda_coord=5.0, lambda_noobj=0.5):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj

    def _iou_for_assignment(self, pred_boxes, gt_boxes):
        # x,y는 cell-relative / w,h는 image-relative이므로 IoU 계산 전 좌표계를 통일.
        gy = torch.arange(self.S, device=pred_boxes.device, dtype=pred_boxes.dtype).view(1, self.S, 1, 1)
        gx = torch.arange(self.S, device=pred_boxes.device, dtype=pred_boxes.dtype).view(1, 1, self.S, 1)

        px = (pred_boxes[..., 0] + gx) / self.S
        py = (pred_boxes[..., 1] + gy) / self.S
        pw = pred_boxes[..., 2].clamp(min=0)
        ph = pred_boxes[..., 3].clamp(min=0)

        tx = (gt_boxes[..., 0] + gx) / self.S
        ty = (gt_boxes[..., 1] + gy) / self.S
        tw = gt_boxes[..., 2].clamp(min=0)
        th = gt_boxes[..., 3].clamp(min=0)

        pxyxy = torch.stack((px-pw/2, py-ph/2, px+pw/2, py+ph/2), -1)
        txyxy = torch.stack((tx-tw/2, ty-th/2, tx+tw/2, ty+th/2), -1)
        inter_lt = torch.maximum(pxyxy[..., :2], txyxy[..., :2])
        inter_rb = torch.minimum(pxyxy[..., 2:], txyxy[..., 2:])
        inter_wh = (inter_rb-inter_lt).clamp(min=0)
        inter = inter_wh[..., 0] * inter_wh[..., 1]
        union = pw*ph + tw*th - inter
        return inter / union.clamp(min=1e-6)

    def forward(self, predictions, target):
        # predictions: (N, 7, 7, 16) / target: (N, 7, 7, 11)
        n = predictions.shape[0]
        assert predictions.shape[1:] == (self.S, self.S, self.C + self.B*5)
        assert target.shape[1:] == (self.S, self.S, self.C + 5)
        pred_cls = predictions[..., :self.C]
        pred_box = predictions[..., self.C:].reshape(n, self.S, self.S, self.B, 5)
        pred_xywh, pred_conf = pred_box[..., :4], pred_box[..., 4]
        target_cls = target[..., :self.C]
        object_mask = target[..., self.C]  # (N, S, S)
        target_xywh = target[..., self.C+1:self.C+5].unsqueeze(-2)

        # 담당 예측기: object가 있는 셀에서 IoU가 최대인 예측 Box
        with torch.no_grad():
            ious = self._iou_for_assignment(pred_xywh.float(), target_xywh.float())
            responsible_idx = ious.argmax(dim=-1)
            conf_target = ious.to(dtype=predictions.dtype).detach()
        responsible_mask = F.one_hot(responsible_idx, num_classes=self.B).to(predictions.dtype)
        responsible_mask = responsible_mask * object_mask.unsqueeze(-1)
        box_mask = responsible_mask.unsqueeze(-1)

        loss_class = ((pred_cls - target_cls).square() * object_mask.unsqueeze(-1)).sum()
        loss_xy = ((pred_xywh[..., :2] - target_xywh[..., :2]).square()
                   * box_mask).sum()
        # 음수 폭/높이 예측에도 gradient가 흐르는 안정적인 signed sqrt
        pred_wh = pred_xywh[..., 2:4]
        pred_wh_sqrt = torch.sign(pred_wh) * torch.sqrt(pred_wh.abs() + 1e-6)
        target_wh_sqrt = torch.sqrt(target_xywh[..., 2:4].clamp(min=0) + 1e-6)
        loss_wh = ((pred_wh_sqrt - target_wh_sqrt).square() * box_mask).sum()
        loss_coord = loss_xy + loss_wh
        loss_obj = ((pred_conf - conf_target).square() * responsible_mask).sum()
        # 배경 셀의 2개 Box + 객체 셀에서 담당하지 않는 Box
        loss_noobj = (pred_conf.square() * (1.0 - responsible_mask)).sum()

        return (self.lambda_coord*loss_coord + loss_obj
                + self.lambda_noobj*loss_noobj + loss_class) / n

# 데이터 없이도 점검할 수 있는 Loss 테스트
_dummy_pred = torch.randn(2, 7, 7, 16, requires_grad=True) * 0.1
_dummy_target = torch.zeros(2, 7, 7, 11)
_dummy_target[0, 2, 3, 0] = 1
_dummy_target[0, 2, 3, 6:11] = torch.tensor([1., 0.5, 0.5, 0.2, 0.3])
_dummy_loss = YoloLoss()(_dummy_pred, _dummy_target)
assert torch.isfinite(_dummy_loss)
_dummy_loss.backward()
print('YOLO Loss 테스트 통과:', round(_dummy_loss.item(), 4))
del _dummy_pred, _dummy_target, _dummy_loss

## 3. 평가 함수 (전 실험 공통)

원본의 평가 함수를 **그대로** 사용합니다. Validation/Test에서 `confidence threshold=0.01`, `NMS IoU=0.5`, `정답 매칭 IoU=0.5`, **VOC 11-point AP**로 mAP@0.5를 계산합니다. 모든 실험은 동일한 `evaluate_map()`을 사용하므로 공정하게 비교할 수 있습니다.

In [ ]:
def iou_torch_eval(box1, box2):
    """ Calculate IoU between two boxes in (x1, y1, x2, y2) format """
    b1_x1, b1_y1, b1_x2, b1_y2 = box1
    b2_x1, b2_y1, b2_x2, b2_y2 = box2

    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area + 1e-6

    return inter_area / union_area

def decode_yolo_output(output, S=7, B=2, C=6, conf_threshold=0.2):
    """ Convert (S, S, B*5+C) tensor to a list of [class, conf, x1, y1, x2, y2] """
    output = output.detach().cpu()

    all_boxes = [] # (class_idx, conf, x1, y1, x2, y2)

    for i in range(S): # grid y
        for j in range(S): # grid x
            # Class
            class_probs = output[i, j, :C] # (C,)
            pred_class_idx = torch.argmax(class_probs).item()
            pred_class_prob = class_probs[pred_class_idx].item()

            # Boxes
            boxes = output[i, j, C:].reshape(B, 5) # (B, 5)

            for b in range(B):
                # (x_cell, y_cell, w_img, h_img, conf)
                box_data = boxes[b]
                confidence = box_data[4].item()

                # class prob * box confidence = final confidence
                final_confidence = pred_class_prob * confidence

                if final_confidence > conf_threshold:
                    # (x_cell, y_cell) -> (x_img, y_img)
                    x_cell, y_cell, w, h = box_data[0:4]
                    x_img = (j + x_cell) / S # 0~1
                    y_img = (i + y_cell) / S # 0~1
                    w_img = w # 0~1
                    h_img = h # 0~1

                    # (x, y, w, h) -> (x1, y1, x2, y2)
                    x1 = x_img - (w_img / 2)
                    y1 = y_img - (h_img / 2)
                    x2 = x_img + (w_img / 2)
                    y2 = y_img + (h_img / 2)

                    all_boxes.append([pred_class_idx, float(final_confidence),
                                      float(x1), float(y1), float(x2), float(y2)])

    return all_boxes

def non_max_suppression(boxes, iou_threshold=0.5):
    """ Perform Non-Max Suppression """
    if not boxes:
        return []

    # Sort by confidence in descending order
    boxes.sort(key=lambda x: x[1], reverse=True)

    final_boxes = []
    while boxes:
        chosen_box = boxes.pop(0)
        final_boxes.append(chosen_box)

        # NMS: Remove boxes of the same class with IoU >= threshold
        boxes = [box for box in boxes
                 if box[0] != chosen_box[0] # if class is different
                 or iou_torch_eval(chosen_box[2:], box[2:]) < iou_threshold # or IoU is low
                ]

    return final_boxes

def visualize_bbox(img_pil, pred_boxes, img_size=(448, 448)):
    """ Visualization function """
    # PIL -> CV2
    img_cv = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    h, w, _ = img_cv.shape

    # Draw Bbox
    for box in pred_boxes:
        class_idx, conf, x1, y1, x2, y2 = box

        # (0~1) coordinates -> (0~W, 0~H)
        x1_px = int(x1 * w)
        y1_px = int(y1 * h)
        x2_px = int(x2 * w)
        y2_px = int(y2 * h)

        # Class color (simply generate color from class index)
        color = (255 * (class_idx % 3), 255 * ((class_idx * 7) % 11) / 10, 255 * ((class_idx * 3) % 7) / 6)

        # Bbox
        cv2.rectangle(img_cv, (x1_px, y1_px), (x2_px, y2_px), color, 2)

        # Label
        label = f"{VOC_CLASSES[class_idx]}: {conf:.2f}"
        cv2.putText(img_cv, label, (x1_px, y1_px - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # CV2 -> RGB (for plt.imshow)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    return img_cv

def visualize(model, test_img_path):
    # Test Image
    test_img_pil = Image.open(test_img_path).convert('RGB')

    transform_viz = transforms.Compose([
        transforms.Resize((448, 448)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    test_tensor = transform_viz(test_img_pil).to(device).unsqueeze(0)

    with torch.no_grad():
        conv_out = model(test_tensor) # (1, 7, 7, 16)

    # (S, S, 16) -> list of [class, conf, x1, y1, x2, y2] (0~1)
    pred_boxes_rel = decode_yolo_output(conv_out[0], C=6, conf_threshold=0.1)
    pred_boxes_nms = non_max_suppression(pred_boxes_rel, iou_threshold=0.5)

    # Visualize (pass original PIL image and NMS results)
    vis_output = visualize_bbox(test_img_pil, pred_boxes_nms)

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(test_img_pil) # Original Image
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("YOLOv1 Prediction (6 Classes)")
    plt.imshow(vis_output) # Image with Bbox drawn
    plt.axis('off')

    plt.show()

In [ ]:
# Fixed six-class VOC 11-point mAP@0.5 evaluation — DO NOT MODIFY.
# `image_ids` is provided separately: val_image_ids during training,
# and the held-out test IDs ONLY in the final evaluation cell.
def evaluate_map(model, image_ids, split_name="Validation", return_details=False):
    if model is None:
        raise ValueError("No model is available for evaluation.")

    NUM_CLASSES = 6
    conf_thresh = 0.01
    iou_thresh_nms = 0.5
    iou_thresh_map = 0.5
    image_ids = list(image_ids)
    assert image_ids, f"{split_name} image list is empty."

    print(f"Calculating {split_name} mAP@0.5 on {len(image_ids)} images...")
    model.eval()
    eval_device = next(model.parameters()).device
    transform_conv = transforms.Compose([
        transforms.Resize((448, 448)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    all_pred_boxes = []  # (image_idx, class_idx, conf, x1, y1, x2, y2)
    all_gt_boxes = []    # (image_idx, class_idx, x1, y1, x2, y2)

    with torch.no_grad():
        for idx, img_name in enumerate(tqdm(image_ids, desc=f"{split_name} mAP")):
            img_path = os.path.join(DATA_PATH, "VOCdevkit", "VOC2012", "JPEGImages", img_name + ".jpg")
            annot_path = os.path.join(DATA_PATH, "VOCdevkit", "VOC2012", "Annotations", img_name + ".xml")
            img = Image.open(img_path).convert("RGB")
            img_w, img_h = img.size
            data = transform_conv(img).unsqueeze(0).to(eval_device)
            output = model(data)

            boxes = decode_yolo_output(output[0], S=7, B=2, C=6, conf_threshold=conf_thresh)
            boxes = non_max_suppression(boxes, iou_threshold=iou_thresh_nms)
            for class_idx, conf, x1, y1, x2, y2 in boxes:
                all_pred_boxes.append([
                    idx, class_idx, conf,
                    x1 * img_w, y1 * img_h, x2 * img_w, y2 * img_h
                ])

            tree = ET.parse(annot_path)
            for obj in tree.findall("object"):
                class_name = obj.find("name").text
                if class_name not in CLASS_TO_IDX:
                    continue
                class_idx = CLASS_TO_IDX[class_name]
                bbox = obj.find("bndbox")
                x1 = float(bbox.find("xmin").text)
                y1 = float(bbox.find("ymin").text)
                x2 = float(bbox.find("xmax").text)
                y2 = float(bbox.find("ymax").text)
                all_gt_boxes.append([idx, class_idx, x1, y1, x2, y2])

    average_precisions = []
    ap_by_class = {}
    for c_idx in range(NUM_CLASSES):
        preds_c = [p for p in all_pred_boxes if p[1] == c_idx]
        gts_c = [g for g in all_gt_boxes if g[1] == c_idx]
        if not gts_c:
            # This is the same exclusion rule as in the provided mAP evaluation code.
            if preds_c:
                average_precisions.append(0.0)
                ap_by_class[VOC_CLASSES[c_idx]] = 0.0
            continue

        gt_counter = {}
        gt_detected = {}
        for g in gts_c:
            img_idx = g[0]
            gt_counter.setdefault(img_idx, []).append(g[2:])
            gt_detected.setdefault(img_idx, []).append(False)

        preds_c.sort(key=lambda x: x[2], reverse=True)
        TP = np.zeros(len(preds_c))
        FP = np.zeros(len(preds_c))
        for p_idx, pred in enumerate(preds_c):
            img_idx = pred[0]
            pred_box = pred[3:]
            if img_idx not in gt_counter:
                FP[p_idx] = 1
                continue
            best_iou, best_gt_idx = 0, -1
            for gt_idx, gt_box in enumerate(gt_counter[img_idx]):
                iou = iou_torch_eval(pred_box, gt_box)
                if iou > best_iou:
                    best_iou, best_gt_idx = iou, gt_idx
            if best_iou >= iou_thresh_map:
                if not gt_detected[img_idx][best_gt_idx]:
                    TP[p_idx] = 1
                    gt_detected[img_idx][best_gt_idx] = True
                else:
                    FP[p_idx] = 1
            else:
                FP[p_idx] = 1

        TP_cumsum = np.cumsum(TP)
        FP_cumsum = np.cumsum(FP)
        recalls = TP_cumsum / len(gts_c)
        precisions = TP_cumsum / np.maximum(TP_cumsum + FP_cumsum, 1e-12)

        ap = 0.0
        for t in np.arange(0., 1.1, 0.1):
            precision = np.max(precisions[recalls >= t]) if np.any(recalls >= t) else 0.0
            ap += precision / 11.0

        average_precisions.append(ap)
        ap_by_class[VOC_CLASSES[c_idx]] = float(ap)
        print(f"{split_name} — {VOC_CLASSES[c_idx]} AP: {ap:.4f}")

    mAP = float(np.mean(average_precisions)) if average_precisions else 0.0
    print(f"{split_name} mAP@0.5 (VOC 11-point) = {mAP:.4f}")
    if return_details:
        return mAP, ap_by_class
    return mAP

## 4. 기본 모델 학습 — 첫 번째 완주

다음 설정으로 **학습 → 매 Epoch Validation mAP 계산 → 최고 Validation mAP 체크포인트 저장**을 자동 수행합니다. 첫 실행은 비교적 적은 Epoch로 시작하고, 성능 개선 단계에서 Epoch 또는 학습률 등을 바꿔 봅니다.

**실습자가 우선 조정할 항목:** `epochs`, `learning_rate`, `batch_size`, `optimizer`, `scale_range_aug`, `color_jitter`. Train 데이터의 증강 외에 데이터 분할·모델 구조·Loss·평가 함수는 바꾸지 않습니다.

In [ ]:
# 기본 학습 설정 (GPU 메모리가 부족하면 batch_size를 1로 줄이세요.)
BASE_CONFIG = {
    'epochs': 3,                          # 빠른 첫 실습: 3 epoch. 충분한 성능을 원하면 증가
    'batch_size': 2,                     # Colab T4를 고려한 보수적 선택
    'optimizer': 'SGD',                  # 'SGD' 또는 'AdamW'
    'learning_rate': 1e-3,
    'momentum': 0.9,                      # SGD일 때 사용
    'weight_decay': 5e-4,
    'scheduler': 'poly',                 # 'poly', 'cosine', 'constant'
    'warmup_iters': 100,
    'warmup_factor': 0.1,
    'power': 0.9,
    'scale_range_aug': (0.7, 1.3),      # Train 전용 증강 강도
    'color_jitter': (0.5, 0.5, 0.5, 0.1), # 밝기, 대비, 채도, 색조
    'amp': True,                         # CUDA mixed precision; 평가 프로토콜은 변경하지 않음
    'num_workers': 2,
}
CHECKPOINT_DIR = Path('./checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)
print('기본 설정:', BASE_CONFIG)

In [ ]:
# 실험마다 같은 seed / split / evaluation 코드를 사용합니다.
def create_optimizer(model, cfg):
    name = cfg['optimizer'].lower()
    if name == 'sgd':
        return optim.SGD(model.parameters(), lr=cfg['learning_rate'],
                         momentum=cfg['momentum'], weight_decay=cfg['weight_decay'])
    if name == 'adamw':
        return optim.AdamW(model.parameters(), lr=cfg['learning_rate'],
                           weight_decay=cfg['weight_decay'])
    raise ValueError("optimizer는 'SGD' 또는 'AdamW'만 지원합니다.")


def create_scheduler(optimizer, cfg, max_iter):
    warmup_iters = min(int(cfg['warmup_iters']), max(0, max_iter - 1))
    def lr_factor(step):
        if warmup_iters and step < warmup_iters:
            return cfg['warmup_factor'] + (1-cfg['warmup_factor'])*(step/warmup_iters)
        progress = min(1.0, (step-warmup_iters)/max(1, max_iter-warmup_iters))
        if cfg['scheduler'] == 'poly':
            return max(0.0, 1.0-progress)**cfg['power']
        if cfg['scheduler'] == 'cosine':
            return 0.5 * (1.0 + np.cos(np.pi*progress))
        if cfg['scheduler'] == 'constant':
            return 1.0
        raise ValueError("scheduler는 'poly', 'cosine', 'constant' 중 하나입니다.")
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)


def train_experiment(name, cfg, init_checkpoint=None):
    if globals().get('_final_test_evaluated', False):
        raise RuntimeError('최종 Test 평가가 이미 끝났습니다. Test 결과를 본 후 모델을 튜닝하지 마세요.')
    reset_seed(SEED)
    train_transform.scale_range_aug = tuple(cfg['scale_range_aug'])
    train_transform.color_jitter = transforms.ColorJitter(*cfg['color_jitter'])

    loader = DataLoader(
        train_dataset, batch_size=int(cfg['batch_size']), shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        num_workers=int(cfg['num_workers']), pin_memory=(device.type == 'cuda'),
        persistent_workers=bool(cfg['num_workers']), drop_last=False,
    )
    model = YOLOv1(S=S, B=B, C=C, pretrained_backbone=(init_checkpoint is None)).to(device)
    if init_checkpoint is not None:
        # 동일 데이터로 이전에 학습한 모델에서 추가로 학습하는 방식 (외부 추가 사전학습 아님)
        state = torch.load(init_checkpoint, map_location='cpu', weights_only=True)
        model.load_state_dict(state, strict=True)
        del state
    criterion = YoloLoss(S=S, B=B, C=C, lambda_coord=5.0, lambda_noobj=0.5)
    optimizer = create_optimizer(model, cfg)
    scheduler = create_scheduler(optimizer, cfg, int(cfg['epochs'])*len(loader))
    use_amp = bool(cfg['amp']) and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best_map = -1.0
    best_epoch = -1
    best_aps = {}
    history = []
    ckpt_path = CHECKPOINT_DIR / f'{name}_best.pth'

    print(f'\n▶ {name} 학습 시작 | 초기 가중치: {init_checkpoint or "ImageNet VGG16"}')
    for epoch in range(1, int(cfg['epochs'])+1):
        model.train()
        running_loss = 0.0
        for images, targets in tqdm(loader, desc=f'{name} {epoch}/{cfg["epochs"]}', leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, targets)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'{name} 학습 중 Loss가 NaN/Inf입니다. 학습률을 낮춰 보세요.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += float(loss.item())

        mean_loss = running_loss/len(loader)
        # Validation: 원본 evaluate_map() / 공식 train에서 분할된 val_image_ids만 사용
        val_map, ap_by_class = evaluate_map(
            model, val_image_ids, split_name=f'{name} Val epoch {epoch}', return_details=True)
        history.append({'epoch': epoch, 'train_loss': mean_loss, 'val_map': val_map})
        print(f'[{name}] Epoch {epoch:02d}/{cfg["epochs"]} | Train Loss {mean_loss:.3f} | Validation mAP@0.5 {val_map:.4f}')
        if val_map > best_map:
            best_map, best_epoch, best_aps = val_map, epoch, ap_by_class.copy()
            # GPU 메모리 추가 복사를 피하고 Colab 디스크에 저장
            torch.save(model.state_dict(), ckpt_path)
            print(f'  ✓ 최고 Validation 성능 갱신 → {ckpt_path}')

    del loader, optimizer, scheduler, scaler, criterion, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'name': name, 'checkpoint': str(ckpt_path),
            'best_val_map': float(best_map), 'best_epoch': best_epoch,
            'best_aps': best_aps, 'history': history, 'config': dict(cfg),
            'init_checkpoint': str(init_checkpoint) if init_checkpoint else None}


def load_model_checkpoint(path):
    # 학습 완료된 checkpoint에 pretrained VGG 파라미터가 포함돼 있으므로 다시 다운로드하지 않습니다.
    model = YOLOv1(S=S, B=B, C=C, pretrained_backbone=False)
    state = torch.load(path, map_location='cpu', weights_only=True)
    model.load_state_dict(state, strict=True)
    del state
    return model.to(device).eval()


def plot_validation_history(run):
    h = pd.DataFrame(run['history'])
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(h['epoch'], h['train_loss'], marker='o')
    axes[0].set(title='Train Loss', xlabel='Epoch', ylabel='Loss')
    axes[1].plot(h['epoch'], h['val_map'], marker='o')
    axes[1].set(title='Validation mAP@0.5', xlabel='Epoch', ylabel='mAP')
    axes[1].set_ylim(bottom=0, top=max(0.05, min(1.0, 1.15*h['val_map'].max())))
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.xaxis.get_major_locator().set_params(integer=True)
    fig.suptitle(run['name'])
    fig.tight_layout()
    plt.show()


def plot_ap(aps, title):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    values = [aps.get(cls, 0.) for cls in VOC_CLASSES]
    ax.bar(VOC_CLASSES, values)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('AP@0.5 (VOC 11-point)')
    ax.set_title(title)
    for i, val in enumerate(values):
        ax.text(i, min(float(val)+0.02, 0.98), f'{val:.3f}', ha='center', fontsize=9)
    fig.tight_layout()
    plt.show()

In [ ]:
# 기본 학습: 이 셀을 실행하면 모든 Train 이미지를 이용한 학습과 매 Epoch 정량 평가를 완료합니다.
results = []
baseline = train_experiment('baseline', BASE_CONFIG)
results.append(baseline)
print('\n★ 기본 실습 결과:')
print(f'최고 Validation mAP@0.5: {baseline["best_val_map"]:.4f} (Epoch {baseline["best_epoch"]})')
plot_validation_history(baseline)
plot_ap(baseline['best_aps'], 'Baseline — Validation AP by Class')

## 5. 정성 평가 — 실제 이미지에 Bounding Box 표시하기

Validation 이미지 3장에 대해 **원본 / 정답 Box(Ground Truth) / 모델 예측 Box**를 비교합니다. 아래 시각화에서는 보기 편하도록 confidence `0.1` 이상의 예측을 표시하지만, **정량 평가의 임계값 0.01은 변경하지 않습니다.**

In [ ]:
# 정성 평가에는 Validation 이미지만 사용합니다 (최종 Test 이미지 접근 금지).
def qualitative_compare(model, image_id, viz_threshold=0.1, max_boxes=15):
    jpg = VOC_DIR / 'JPEGImages' / f'{image_id}.jpg'
    xml = VOC_DIR / 'Annotations' / f'{image_id}.xml'
    image = Image.open(jpg).convert('RGB')
    original = np.asarray(image).copy()
    gt_vis = original.copy()
    pred_vis = original.copy()
    h, w = original.shape[:2]

    root = ET.parse(xml).getroot()
    for obj in root.findall('object'):
        label = obj.find('name').text
        if label not in CLASS_TO_IDX:
            continue
        box = obj.find('bndbox')
        x1, y1, x2, y2 = [int(float(box.find(k).text)) for k in ('xmin','ymin','xmax','ymax')]
        cv2.rectangle(gt_vis, (x1,y1), (x2,y2), (0,185,35), 2)
        cv2.putText(gt_vis, label, (x1,max(12,y1-5)), cv2.FONT_HERSHEY_SIMPLEX,
                    .5, (0,185,35), 2)

    inp = transforms.Compose([
        transforms.Resize((448,448)), transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(inp)[0]
    boxes = decode_yolo_output(output, S=7, B=2, C=6, conf_threshold=viz_threshold)
    boxes = non_max_suppression(boxes, iou_threshold=0.5)[:max_boxes]
    for cls, confidence, x1, y1, x2, y2 in boxes:
        a = (int(x1*w),int(y1*h))
        b = (int(x2*w),int(y2*h))
        cv2.rectangle(pred_vis, a, b, (220,40,40), 2)
        cv2.putText(pred_vis, f'{VOC_CLASSES[cls]} {confidence:.2f}',
                    (a[0],max(12,a[1]-5)), cv2.FONT_HERSHEY_SIMPLEX, .5, (220,40,40), 2)

    fig, axes = plt.subplots(1,3,figsize=(16,4))
    for ax, img, title in zip(axes, [original,gt_vis,pred_vis],
                              ['Original', 'Ground Truth', 'YOLOv1 Prediction']):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'Validation image: {image_id} | Visualization threshold: {viz_threshold}')
    fig.tight_layout()
    plt.show()

baseline_model = load_model_checkpoint(baseline['checkpoint'])
for image_id in val_image_ids[:3]:
    qualitative_compare(baseline_model, image_id)
del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. 성능 개선 실습 — 조건을 바꾸고 비교하기

앞에서 얻은 기본 모델의 **최고 Validation 체크포인트**부터 짧게 **추가 학습**하는 3개 실험을 준비했습니다. 같은 출발점에서 한 가지 설정씩 바꾸면 차이를 이해하기 쉽습니다. 반복 실험도 **Train/Validation만** 사용하며, 최종 모델 선택 기준은 **최고 Validation mAP@0.5**입니다.

- **실험 A:** 학습 Epoch 추가 (같은 설정으로 더 학습)
- **실험 B:** Learning Rate 줄이기 (`1e-3 → 3e-4`)
- **실험 C:** Augmentation 약하게 하기 (`scale 0.7~1.3 → 0.85~1.15`)

기본 실행에서는 시간이 추가로 소요되지 않도록 이 단계를 건너뜁니다. 실습 시 `RUN_EXTRA_EXPERIMENTS=True`로 바꾸고, 원하는 실험 이름만 `SELECTED_EXPERIMENTS`에 남겨 실행하세요. **최종 Test를 수행하기 전**에 이 실험들을 마쳐야 합니다.

In [ ]:
RUN_EXTRA_EXPERIMENTS = False          # <- 성능 개선 실습에서 True로 변경
SELECTED_EXPERIMENTS = ['more_epochs', 'lower_lr', 'milder_augmentation']

# 모든 추가 실험은 baseline best 모델에서 시작; 고정 seed=42, 데이터·평가 Protocol 불변.
EXPERIMENT_LIBRARY = {
    'more_epochs': {
        'epochs': 2,                # Baseline best에서 2 Epoch 더 학습
    },
    'lower_lr': {
        'epochs': 2,
        'learning_rate': 3e-4,     # 학습률만 변경
    },
    'milder_augmentation': {
        'epochs': 2,
        'scale_range_aug': (0.85, 1.15),  # Train 증강 범위만 변경
    },
}

if RUN_EXTRA_EXPERIMENTS:
    for trial_name in SELECTED_EXPERIMENTS:
        assert trial_name in EXPERIMENT_LIBRARY, f'정의되지 않은 실험: {trial_name}'
        new_cfg = {**BASE_CONFIG, **EXPERIMENT_LIBRARY[trial_name]}
        # 동일한 실험을 다시 수행하면 새 버전으로 저장해 이전 결과를 보존합니다.
        run_name = trial_name
        version = 2
        existing_names = {r['name'] for r in results}
        while run_name in existing_names:
            run_name = f'{trial_name}_v{version}'
            version += 1
        trial = train_experiment(run_name, new_cfg, init_checkpoint=baseline['checkpoint'])
        results.append(trial)
        plot_validation_history(trial)
else:
    print('기본 실습 완료. 추가 실험을 하려면 RUN_EXTRA_EXPERIMENTS=True로 변경하세요.')

## 7. 실험 결과 비교 및 최종 모델 선택

**Test 결과가 아닌 Validation mAP@0.5**만 기준으로 가장 좋은 체크포인트를 선택합니다. 이 결과표는 설정을 바꾸어 여러 번 실험할 때 누적 비교하는 용도입니다.

In [ ]:
comparison = pd.DataFrame([{
    '실험': r['name'],
    '최고 Val mAP@0.5': r['best_val_map'],
    'Best Epoch': r['best_epoch'],
    '추가 학습 Epoch': r['config']['epochs'],
    'Learning Rate': r['config']['learning_rate'],
    'Optimizer': r['config']['optimizer'],
    'Scale Augmentation': str(r['config']['scale_range_aug']),
} for r in results]).sort_values('최고 Val mAP@0.5', ascending=False).reset_index(drop=True)
display(comparison)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(comparison['실험'], comparison['최고 Val mAP@0.5'])
ax.set(title='Validation mAP@0.5 — Experiment Comparison', ylabel='mAP@0.5', ylim=(0,1.0))
ax.tick_params(axis='x', labelrotation=15)
fig.tight_layout()
plt.show()

best_run = max(results, key=lambda r: r['best_val_map'])
print('최종 선택 (Validation 기준):', best_run['name'])
print('Best Validation mAP@0.5   :', f'{best_run["best_val_map"]:.4f}')
print('Checkpoint                :', best_run['checkpoint'])

## 8. 최종 Test 평가 — 설정을 모두 정한 뒤 한 번만 실행

공식 VOC 2012 `val.txt` 이미지 전체를 **마지막에 한 번만** 평가합니다. 최종 설정을 바꾸기 전까지는 `RUN_FINAL_TEST=False`를 유지하세요. **모든 추가 실험을 마치면 아래 셀을 `True`로 바꾸어 해당 셀만 실행**하면 됩니다.

평가 결과: **Test mAP@0.5, 클래스별 AP 그래프**, 최종 가중치 파일 `YOLOv1_best.pth`.

In [ ]:
RUN_FINAL_TEST = False  # <- 모든 실험이 끝난 뒤 True로 바꾸고 이 셀만 실행

if not RUN_FINAL_TEST:
    print('최종 Test는 아직 수행하지 않았습니다.')
    print('Validation 기반 실험을 모두 마친 뒤 RUN_FINAL_TEST=True로 바꾸세요.')
else:
    assert not globals().get('_final_test_evaluated', False), '이 런타임에서 최종 Test를 이미 평가했습니다.'
    assert best_run is not None and Path(best_run['checkpoint']).is_file()
    final_test_list_path = VOC_DIR / 'ImageSets' / 'Main' / 'val.txt'
    # 공식 VOC val.txt는 이 단계에서 처음 읽습니다.
    with open(final_test_list_path, 'r', encoding='utf-8') as f:
        final_test_ids = [line.strip() for line in f if line.strip()]
    assert not (set(final_test_ids) & set(val_image_ids)), 'Validation과 Test가 겹칩니다.'
    assert not (set(final_test_ids) & set(Path(train_full_aug.images[i]).stem for i in train_indices)), 'Train과 Test가 겹칩니다.'

    model_path = Path('YOLOv1_best.pth')
    shutil.copyfile(best_run['checkpoint'], model_path)
    final_model = load_model_checkpoint(model_path)
    # 평가 코드는 원본과 동일: conf=0.01, NMS IoU=0.5, matching IoU=0.5, VOC 11-point AP.
    final_test_map, test_ap_by_class = evaluate_map(
        final_model, final_test_ids, split_name='Final Test', return_details=True)
    _final_test_evaluated = True

    print('\n' + '='*65)
    print('최종 선택 실험           :', best_run['name'])
    print('Best Validation mAP@0.5 :', f'{best_run["best_val_map"]:.4f}')
    print('Final Test mAP@0.5      :', f'{final_test_map:.4f}')
    print('가중치 저장             :', model_path.resolve())
    print('클래스별 Test AP')
    for cls in VOC_CLASSES:
        print(f'  {cls:>8} : {test_ap_by_class.get(cls, float("nan")):.4f}')
    print('='*65)
    plot_ap(test_ap_by_class, 'Final Test — AP by Class')
    del final_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()